# 03 — Baselines: 12-month logistic and Cox

**Goal:** Fit two simple default models on training loans, compare how well they rank and how well their probabilities match reality on validation, and decide what to do next. Look at test once, at the end.

## Setup

Both models use the same six features and the same preprocessing, fit on training loans only.

In [ ]:
import pandas as pd

from mortgage_credit_risk.config import load_config
from mortgage_credit_risk.evaluate import (
    binary_metrics,
    c_index,
    calibration_table,
    hazard_ratios_by_window,
    paired_bootstrap_auc,
)
from mortgage_credit_risk.models import (
    coefficients,
    default_by,
    fit_cox,
    fit_logistic,
    hazard_ratios,
    horizon_label,
)
from mortgage_credit_risk.pipeline import build_model_tables

cfg = load_config()
_, loans_path = build_model_tables(cfg, cfg.build_rate)
loans = pd.read_parquet(loans_path)
FEATURES = [*cfg.numeric_features, *cfg.categorical_features]
H = cfg.logistic_horizon_months

train = loans[loans["split"] == "train"]
val = loans[loans["split"] == "validation"]
test = loans[loans["split"] == "test"]
labeled = horizon_label(loans, H)

logit = fit_logistic(labeled[labeled["split"] == "train"], cfg)
cox = fit_cox(train, cfg)

## The 12-month label

Logistic regression needs a yes or no answer: default by loan age 12. Loans watched for less than 12 months with no event have no answer and are left out. Cox uses every loan for as long as it was watched.

In [ ]:
pd.DataFrame(
    {
        "loans": loans.groupby("split").size(),
        "kept_for_logistic": labeled.groupby("split").size(),
        "defaults_by_12": labeled.groupby("split")["target"].sum(),
        "defaults_any_age": loans.assign(d=loans["event"] == 1)
        .groupby("split")["d"]
        .sum(),
    }
)

## What the models learned

Numeric inputs are scaled, so each ratio is per one training standard deviation. Categories are compared with cash-out (purpose) and investor (occupancy).

In [ ]:
pd.DataFrame(
    {
        "logistic_odds_ratio": coefficients(logit)["odds_ratio"].drop("intercept"),
        "cox_hazard_ratio": hazard_ratios(cox)["hazard_ratio"],
    }
).round(2)

## Proportional hazards check

Cox assumes each hazard ratio is the same at every loan age. Fit Cox separately on young loans and on loans still active at the split age, and compare.

In [ ]:
window = hazard_ratios_by_window(train, cox.named_steps["pre"], cfg.ph_split_age)
window.reset_index().pivot(
    index="feature", columns="window", values=["hazard_ratio", "low_95", "high_95"]
).round(2)

## Validation: ranking

AUC on the 12-month label, and C-index on all validation loans using every month they were watched.

In [ ]:
v12 = labeled[labeled["split"] == "validation"]
p_logit = logit.predict_proba(v12[FEATURES])[:, 1]
p_cox = default_by(cox, v12, H)

pd.DataFrame(
    {
        "auc_12m": {
            "logistic": binary_metrics(v12["target"], p_logit)["auc"],
            "cox": binary_metrics(v12["target"], p_cox)["auc"],
        },
        "c_index_all_months": {
            "logistic": c_index(val, logit.predict_proba(val[FEATURES])[:, 1]),
            "cox": c_index(val, cox.predict(val[FEATURES])),
        },
    }
).round(3)

Is the Cox advantage real? Both models are scored on the same resampled loans, so shared noise cancels in the difference.

In [ ]:
pd.Series(
    paired_bootstrap_auc(v12["target"], p_cox, p_logit, cfg.n_bootstrap, cfg.seed)
).round(4)

## Validation: calibration

Predicted and observed default rates by loan age 12, in percent. A ratio below 1 means the model predicts too few defaults.

In [ ]:
summary = pd.DataFrame(
    {
        "logistic": binary_metrics(v12["target"], p_logit),
        "cox": binary_metrics(v12["target"], p_cox),
    }
).T
summary[["mean_predicted", "observed"]] *= 100
summary.round(4)

In [ ]:
bands = pd.concat(
    {
        "logistic": calibration_table(v12["target"], p_logit, cfg.n_bins),
        "cox": calibration_table(v12["target"], p_cox, cfg.n_bins),
    },
    axis=1,
)
for model in ["logistic", "cox"]:
    bands[(model, "mean_predicted")] *= 100
    bands[(model, "observed")] *= 100
bands.round(3)

## Test: one look

No choice in this notebook depends on these numbers. Test loans are the 2019 vintage, followed to Mar 2026.

In [ ]:
t12 = labeled[labeled["split"] == "test"]
t_logit = logit.predict_proba(t12[FEATURES])[:, 1]
t_cox = default_by(cox, t12, H)
result = pd.DataFrame(
    {
        "logistic": binary_metrics(t12["target"], t_logit),
        "cox": binary_metrics(t12["target"], t_cox),
    }
).T
result[["mean_predicted", "observed"]] *= 100
result["c_index_all_months"] = [
    c_index(test, logit.predict_proba(test[FEATURES])[:, 1]),
    c_index(test, cox.predict(test[FEATURES])),
]
result.round(4)

## Findings

Numbers are for the 5% build sample.

1. **Cox uses much more of the data.** The 12-month logistic model learns from 169 training defaults. Cox learns from all 884, because it uses every loan for as long as it was watched.

2. **Credit score is the strongest feature in both models.** One standard deviation higher (about 44 points) cuts default odds by 58% in logistic and the monthly default risk by 54% in Cox.

3. **Cox finds effects the 12-month window hides.** DTI (hazard ratio 1.32) and LTV (1.20) are clearly above 1 in Cox, with intervals 1.22 to 1.43 and 1.09 to 1.32. In logistic they look small (1.09 and 1.05) and their intervals include 1. Most defaults happen after month 12.

4. **Owner-occupied loans default more than investor loans.** Hazard ratio 1.67 (1.20 to 2.33). Raw rates agree: 0.64% vs 0.38%. Freddie Mac screens investor loans harder, and investors who want out tend to sell or refinance, which counts as censoring here.

5. **Proportional hazards mostly holds.** Hazard ratios for young loans (under 18 months) and older loans are close for every feature except occupancy. The owner vs investor ratio grows from 1.19 to 2.16. The intervals overlap, so this is a warning, not proof.

6. **Cox ranks a little better, and the gap is real.** On validation, Cox has AUC 0.812 vs 0.800 and C-index 0.815 vs 0.805. In a paired bootstrap, Cox wins in 99.8% of resamples, with an AUC gain of 0.004 to 0.020.

7. **Both models predict about 30% too few defaults on validation.** Predicted vs observed 12-month default rate is 0.162% vs 0.240% (logistic) and 0.171% vs 0.240% (Cox). On training the logistic ratio is 1.01, so this is not underfitting. Validation loans and years were riskier in ways the six features do not capture.

8. **Ranking drops sharply on test.** Test AUC is 0.666 (logistic) and 0.675 (Cox), down from about 0.80 on validation. C-index is 0.70 and 0.72. Test loans are the 2019 vintage, so their first year ran into COVID. Who got into trouble in 2020 depended a lot on jobs and industry, which origination features cannot see. Calibration on test is the same as on validation: about 30% too low.

9. **The same story holds on validation and test: the period matters more than the model.** Two different models fail in the same way. That points to missing information about the economy, not the model type.

**Decision:**

- Keep Cox as the main baseline. It uses more data, ranks better, and gives a default curve over time.
- Next: add economic features from FRED (market mortgage rate, unemployment, house prices), starting with the rate spread at origination. Then rerun both baselines and check whether the predicted vs observed ratio moves toward 1.
- No CatBoost and no recalibration for now. The gap comes from missing information about the period, which neither fixes. The 30% under-prediction must be addressed before expected loss (M6).